# Vector Database

Make sure you have a .env file with the following fields: 

- PGHOST=localhost
- PGPORT=5432
- PGDB=rag_db
- PGUSER=postgres
- PGPASSWORD=your_postgres_password

## Import libraries and define paths

In [5]:
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Iterator, Dict, Any, List
import numpy as np

# To import the passwords
import os
#pip install python-dotenv
from dotenv import load_dotenv

# To connect with posgreSQL db
import psycopg2
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values

In [ ]:
DOCS_ROOT = Path("docs")
TOPIC_FOLDERS = {"general", "mama"}  # extend later: {"general","mama","prostata",...}
PAGES_JSONL = Path("docs/pages.jsonl")
CHUNKS_JSONL = Path("docs/chunks.jsonl")
#Document metadata structure
@dataclass
class DocMeta:
    doc_id: str #stable unique ID
    topic: str #mama, general, prostata, etc.
    lang: str #language
    source: str #novartis, gepac, etc.
    slug: str #descriptive text
    version: str #version (v1, v2) or year of publication
    path: str #where the file is on disk
    file_hash: str #unique hash to detect changes

In [8]:
load_dotenv(".env")  # path to your secrets file
dbname=os.environ["PGDB"]
user=os.environ["PGUSER"]
password=os.environ["PGPASSWORD"]
host=os.environ['PGHOST']
port=os.environ['PGHOST']


## Load chunk text and chunk vectors

In [ ]:
emb = np.load("docs/chunks_vectors.npy")
emb.shape #(909, 384)

In [ ]:
loaded_chunks = []
with CHUNKS_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        loaded_chunks.append(json.loads(line))

print("Loaded chunks:", len(loaded_chunks)) #909

## Add embeddings to PostgreSQL vector db (bulk insert)

In [ ]:
# 1) Connect to YOUR database (not "postgres" unless that's where your table is)
conn = psycopg2.connect(
    dbname=dbname,          # <-- change to your DB name
    user=user,
    password=password, #Don't upload to GitHub
    host=host,
    port=port
)

# 2) Make psycopg2 understand the pgvector type
register_vector(conn)

cur = conn.cursor()

# 3) Suppose you already have:
# texts = [c["text"] for c in all_chunks]
# emb = emb_model.encode(...)

# emb is usually a numpy array shape (N, 384).
# Convert each row to a plain Python list (pgvector adapter handles it well)´
texts = loaded_chunks.copy()
#rows = [(t, e.tolist()) for t, e in zip(texts, emb)]
# loaded_chunks is a list of dicts, emb is (N, 384)
rows = [(c["text"], emb[i].tolist()) for i, c in enumerate(loaded_chunks)]
print(rows[1])

# 4) Bulk insert (fast)
'''Assuming that you table structure is: 
    CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT,
    embedding VECTOR(1536)
    );
'''
execute_values(
    cur,
    "INSERT INTO documents (content, embedding) VALUES %s",
    rows
)

conn.commit()
cur.close()
conn.close()

print(f"Inserted {len(rows)} rows.")

### Sanity check

In [9]:
conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host)
register_vector(conn)
cur = conn.cursor()

cur.execute("SELECT id, left(content, 400), embedding FROM documents LIMIT 2;")
print(cur.fetchall())

cur.close()
conn.close()

[(1, 'Vol.:(0123456789) 1 3 Clinical and Translational Oncology (2023) 25:2665–2678 https://doi.org/10.1007/s12094-023-03203-8 CLINICAL GUIDES IN\xa0ONCOLOGY SEOM–GEICAM–SOLTI clinical guidelines in\xa0advanced breast cancer (2022) Jose\xa0Angel\xa0Garcia‑Saenz1\u200a \xa0· Isabel\xa0Blancas2\u200a \xa0· Isabel\xa0Echavarria3\u200a \xa0· Carmen\xa0Hinojo4\u200a \xa0· Mireia\xa0Margeli5\xa0· Fernando\xa0Moreno1\u200a \xa0· Sonia\xa0Pernas6\u200a \xa0· Teresa\xa0Ramon\xa0y\xa0Cajal7\u200a \xa0· Nuria\xa0', array([-5.79986349e-02, -8.59923959e-02,  1.97257027e-02,  3.19715515e-02,
        9.42024309e-03, -5.08584431e-04, -3.04730274e-02,  5.03634773e-02,
        2.17071306e-02, -4.89296839e-02,  9.60904581e-04, -3.74150351e-02,
       -6.54999912e-02,  1.22926245e-02,  8.46127421e-03, -1.27214612e-02,
       -2.88276747e-02, -3.69683392e-02, -6.94779083e-02,  4.71994542e-02,
       -5.87821864e-02,  9.24992934e-03,  2.94780955e-02,  3.47960629e-02,
       -4.33659367e-02, -1.47012085e-01,

In [10]:
conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host)
register_vector(conn)
cur = conn.cursor()

cur.execute("SELECT id, content, embedding FROM documents WHERE id = 5;")
print(cur.fetchall())

cur.close()
conn.close()

[(5, 'In patients with metastatic triple negative breast cancer (TNBC), programmed death-ligand 1 (PD-L1) status should be determined by immunohistochemistry, to decide if therapy with immune checkpoint inhibitors should be incorporated to first-line treatment [I, A].\n\nIn HER2-negative MBC, germline BRCA1/2 mutations (gBRCAm) status should be tested since treatment with PARP inhibitors could be indicated [I, A].\n\nSomatic sequencing [II-A] cannot fully substitute germline BRCA testing but could guide to confirm a potential gBRCAm status.\n\nIn ER and/or PR positive HER2-negative ABC patients, phosphatidylinositol-4,5-bisphosphate 3-kinase catalytic subunit alpha (PIK3CA) mutations should be assessed [I-A] to consider the use of PIK3CA inhibitors [3].\n\nTable\u202f1\u2002 \u2009Strength of recommendation and quality of evidence score Category, grade Definition Strength of recommendation A Good evidence to support a recommendation for use B Moderate evidence to support a recommendati